In [ ]:
!pip install ultralytics
!pip install scikit-learn
!pip install timm
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 25.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/archive (6).zip" -d /content/

In [ ]:
!ls /content/agri_data/data | head

agri_0_1009.jpeg
agri_0_1009.txt
agri_0_1017.jpeg
agri_0_1017.txt
agri_0_1018.jpeg
agri_0_1018.txt
agri_0_1020.jpeg
agri_0_1020.txt
agri_0_1024.jpeg
agri_0_1024.txt


In [ ]:
import os, shutil, random

src = "/content/agri_data/data"

images = [f for f in os.listdir(src) if f.endswith(".jpg")]

random.shuffle(images)

split = int(0.8 * len(images))

train = images[:split]
val = images[split:]

for p in [
"/content/dataset/images/train",
"/content/dataset/images/val",
"/content/dataset/labels/train",
"/content/dataset/labels/val"]:
    os.makedirs(p, exist_ok=True)

for img in train:

    label = img.replace(".jpg",".txt")

    shutil.copy(f"{src}/{img}", f"/content/dataset/images/train/{img}")
    shutil.copy(f"{src}/{label}", f"/content/dataset/labels/train/{label}")

for img in val:

    label = img.replace(".jpg",".txt")

    shutil.copy(f"{src}/{img}", f"/content/dataset/images/val/{img}")
    shutil.copy(f"{src}/{label}", f"/content/dataset/labels/val/{label}")

In [ ]:
%%writefile agri.yaml
path: /content/dataset
train: images/train
val: images/val

names:
  0: crop
  1: weed

Writing agri.yaml


In [ ]:
import torch
import torch.nn as nn

class AgroKDNet(nn.Module):

    def __init__(self,num_classes=2):

        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(3,32,3,2,1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,64,3,2,1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,128,3,2,1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128,256,3,2,1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        self.head = nn.Conv2d(256,num_classes+5,1)

    def forward(self,x):

        x = self.backbone(x)

        x = self.head(x)

        return x

In [ ]:
!ls -R /content | head -50

/content:
agri_data
agri.yaml
classes.txt
dataset
drive
sample_data

/content/agri_data:
data

/content/agri_data/data:
agri_0_1009.jpeg
agri_0_1009.txt
agri_0_1017.jpeg
agri_0_1017.txt
agri_0_1018.jpeg
agri_0_1018.txt
agri_0_1020.jpeg
agri_0_1020.txt
agri_0_1024.jpeg
agri_0_1024.txt
agri_0_1026.jpeg
agri_0_1026.txt
agri_0_1028.jpeg
agri_0_1028.txt
agri_0_1041.jpeg
agri_0_1041.txt
agri_0_1053.jpeg
agri_0_1053.txt
agri_0_1068.jpeg
agri_0_1068.txt
agri_0_106.jpeg
agri_0_106.txt
agri_0_1079.jpeg
agri_0_1079.txt
agri_0_1082.jpeg
agri_0_1082.txt
agri_0_1083.jpeg
agri_0_1083.txt
agri_0_1094.jpeg
agri_0_1094.txt
agri_0_1095.jpeg
agri_0_1095.txt
agri_0_110.jpeg
agri_0_110.txt
agri_0_1114.jpeg
agri_0_1114.txt
agri_0_1119.jpeg
agri_0_1119.txt


In [ ]:
import os

src = "/content/agri_data/data"

images = [f for f in os.listdir(src) if f.lower().endswith(".jpeg")]
print("Total images:", len(images))

Total images: 1300


In [ ]:
import os
import shutil
import random

src = "/content/agri_data/data"

files = os.listdir(src)

images = [f for f in files if f.lower().endswith((".jpg",".jpeg",".png"))]

print("Total images:", len(images))

random.shuffle(images)

split = int(0.8 * len(images))

train = images[:split]
val = images[split:]

dirs = [
"/content/dataset/images/train",
"/content/dataset/images/val",
"/content/dataset/labels/train",
"/content/dataset/labels/val"
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

def copy_pair(img, target):

    name = os.path.splitext(img)[0]
    label = name + ".txt"

    shutil.copy(os.path.join(src,img),
                f"/content/dataset/images/{target}/{img}")

    label_path = os.path.join(src,label)

    if os.path.exists(label_path):
        shutil.copy(label_path,
                    f"/content/dataset/labels/{target}/{label}")

for img in train:
    copy_pair(img,"train")

for img in val:
    copy_pair(img,"val")

Total images: 1300


In [ ]:
!head /content/dataset/labels/train/*.txt

==> /content/dataset/labels/train/agri_0_1017.txt <==
1 0.496094 0.384766 0.960938 0.753906

==> /content/dataset/labels/train/agri_0_1018.txt <==
1 0.445312 0.483398 0.808594 0.634766

==> /content/dataset/labels/train/agri_0_1020.txt <==
1 0.494141 0.350586 0.730469 0.611328

==> /content/dataset/labels/train/agri_0_1028.txt <==
0 0.476562 0.285156 0.070312 0.062500
0 0.480469 0.169922 0.054688 0.082031
0 0.148438 0.235352 0.070312 0.091797
0 0.420898 0.489258 0.064453 0.041016
0 0.767578 0.408203 0.042969 0.066406
0 0.742188 0.637695 0.082031 0.064453
0 0.446289 0.706055 0.068359 0.056641
0 0.542969 0.812500 0.050781 0.066406
0 0.764648 0.923828 0.064453 0.062500
0 0.166992 0.817383 0.083984 0.044922

==> /content/dataset/labels/train/agri_0_1041.txt <==
0 0.513672 0.500977 0.785156 0.998047

==> /content/dataset/labels/train/agri_0_1053.txt <==
0 0.538086 0.448242 0.712891 0.759766

==> /content/dataset/labels/train/agri_0_1068.txt <==
1 0.375977 0.464844 0.611328 0.648438

==> /co

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="agri.yaml",
    epochs=10,
    imgsz=512,
    batch=16,
    device=0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=agri.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, kera

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78222833e5a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
metrics = model.val()

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1544.7±747.2 MB/s, size: 64.1 KB)
val: Scanning /content/dataset/labels/val.cache... 260 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 260/260 83.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 4.2it/s 4.0s
                   all        260        395      0.835      0.783      0.864      0.581
                  crop        124        219      0.765      0.808      0.884      0.632
                  weed        136        176      0.905      0.757      0.844       0.53
Speed: 3.4ms preprocess, 3.9ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/runs/detect/val


In [ ]:
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

f1 = 2 * (precision*recall)/(precision+recall)

iou = metrics.box.maps.mean()*100

print("mAP@0.5:",map50)
print("mAP@0.5:0.95:",map5095)
print("Precision:",precision)
print("Recall:",recall)
print("F1 Score:",f1)
print("IoU %:",iou)

mAP@0.5: 0.8639383953559439
mAP@0.5:0.95: 0.5811723398699541
Precision: 0.8347653882585488
Recall: 0.7825561295388077
F1 Score: 0.8078180672427305
IoU %: 58.11723398699541


In [ ]:
def print_metrics(metrics):

    precision = metrics.box.mp
    recall = metrics.box.mr
    map50 = metrics.box.map50
    map5095 = metrics.box.map

    f1 = 2*(precision*recall)/(precision+recall)
    iou = metrics.box.maps.mean()*100

    print("mAP@0.5:", map50)
    print("mAP@0.5:0.95:", map5095)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("IoU %:", iou)

    return map50, map5095, precision, recall, f1, iou

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="agri.yaml",
    epochs=5,
    imgsz=512,
    batch=16
)

metrics = model.val()

print("\nYOLOv8-Nano Results")
print_metrics(metrics)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=agri.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=

(np.float64(0.8502865497867909),
 np.float64(0.5355628789473962),
 np.float64(0.823917773354157),
 np.float64(0.7699252801992528),
 np.float64(0.7960070109746052),
 np.float64(53.556287894739626))

In [ ]:
model = YOLO("yolo11n.pt")

model.train(
    data="agri.yaml",
    epochs=5,
    imgsz=512
)

metrics = model.val()

print("\nYOLOv11-Nano Results")
print_metrics(metrics)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=agri.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=

(np.float64(0.8334060630781335),
 np.float64(0.5353831556396682),
 np.float64(0.8609991064294202),
 np.float64(0.7104616412300149),
 np.float64(0.778520035148839),
 np.float64(53.53831556396681))

In [ ]:
!ls

agri_data  classes.txt	drive  sample_data  yolo26n.pt	    yolov8n.pt
agri.yaml  dataset	runs   yolo11n.pt   yolov7-tiny.pt


In [ ]:
!pwd

/content/yolov7/detr/nanodet


In [ ]:
%cd /content/yolov7

/content/yolov7


In [ ]:
!ls

cfg	   export.py   models		 runs	       train.py
data	   figure      paper		 scripts       utils
deploy	   hubconf.py  __pycache__	 test.py       yolov7-tiny.pt
detect.py  inference   README.md	 tools
detr	   LICENSE.md  requirements.txt  train_aux.py


In [ ]:
%cd /content
!git clone https://github.com/WongKinYiu/yolov7
%cd yolov7

!pip install -r requirements.txt

/content
fatal: destination path 'yolov7' already exists and is not an empty directory.
/content/yolov7
  Using cached numpy-1.23.5.tar.gz (10.7 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
!pip install torch torchvision torchaudio
!pip install opencv-python matplotlib tqdm pyyaml scipy

In [ ]:
!wget https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt

--2026-03-06 18:57:28--  https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/511187726/ba7d01ee-125a-4134-8864-fa1abcbf94d5?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-06T19%3A53%3A04Z&rscd=attachment%3B+filename%3Dyolov7-tiny.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-06T18%3A52%3A13Z&ske=2026-03-06T19%3A53%3A04Z&sks=b&skv=2018-11-09&sig=%2FM6A3wq48nlGjK8gCU7MeFvB2j4A6jHpcw1mqh6cFhU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjgyNTI0OSwibmJmIjoxNzcyODIzNDQ5LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLm

In [ ]:
# !git clone https://github.com/RangiLyu/nanodet
# %cd nanodet

# !pip install -r requirements.txt

Cloning into 'nanodet'...
remote: Enumerating objects: 2722, done.
remote: Total 2722 (delta 0), reused 0 (delta 0), pack-reused 2722 (from 1)
Receiving objects: 100% (2722/2722), 5.29 MiB | 8.00 MiB/s, done.
Resolving deltas: 100% (1602/1602), done.
/content/yolov7/detr/nanodet
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement torch<2.0,>=1.10 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch<2.0,>=1.10


In [ ]:
# !wget https://github.com/RangiLyu/nanodet/releases/download/v1.0/nanodet_m.pt

--2026-03-06 18:27:04--  https://github.com/RangiLyu/nanodet/releases/download/v1.0/nanodet_m.pt
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-06 18:27:04 ERROR 404: Not Found.



In [ ]:
# %cd /content
# !git clone https://github.com/RangiLyu/nanodet.git
# %cd nanodet

/content
Cloning into 'nanodet'...
remote: Enumerating objects: 2722, done.
remote: Total 2722 (delta 0), reused 0 (delta 0), pack-reused 2722 (from 1)
Receiving objects: 100% (2722/2722), 5.29 MiB | 11.58 MiB/s, done.
Resolving deltas: 100% (1602/1602), done.
/content/nanodet


In [ ]:
# !pip install -r requirements.txt
# !pip install pycocotools

  Using cached onnx_simplifier-0.5.0.tar.gz (728 bytes)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pyaml-26.2.1-py3-none-any.whl.metadata (12 kB)
  Using cached pytorch_lightning-1.9.5-py3-none-any.whl.metadata (23 kB)
ERROR: Could not find a version that satisfies the requirement torch<2.0,>=1.10 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch<2.0,>=1.10


In [ ]:
# !pip install pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 22.4 MB/s eta 0:00:00


In [ ]:
# !pip install pycocotools
# !pip install tensorboard

In [ ]:
import os
import json
from PIL import Image

def create_coco_json(dataset_path, label_path, output_json_path, categories):

    images_list = []
    annotations = []
    ann_id = 0

    for img_id, img_file in enumerate(os.listdir(dataset_path)):

        if not img_file.lower().endswith((".jpg", ".jpeg", ".png", ".gif")):
            continue

        img_path = os.path.join(dataset_path, img_file)
        try:
            with Image.open(img_path) as img:
                w, h = img.size
        except Exception as e:
            print(f"Could not open image {img_file}: {e}")
            continue

        images_list.append({
            "id": img_id,
            "file_name": img_file,
            "width": w,
            "height": h
        })

        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_file_path = os.path.join(label_path, label_file)

        if os.path.exists(label_file_path):
            with open(label_file_path) as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        c, x, y, bw, bh = map(float, parts)

                        # Convert YOLO format to COCO format (x_min, y_min, width, height)
                        x1 = (x - bw / 2) * w
                        y1 = (y - bh / 2) * h
                        bbox_width = bw * w
                        bbox_height = bh * h

                        annotations.append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": int(c),
                            "bbox": [x1, y1, bbox_width, bbox_height],
                            "area": bbox_width * bbox_height,
                            "iscrowd": 0
                        })

                        ann_id += 1

    coco_data = {
        "images": images_list,
        "annotations": annotations,
        "categories": categories
    }

    with open(output_json_path, "w") as f:
        json.dump(coco_data, f, indent=4)
    print(f"Generated {output_json_path} with {len(images_list)} images and {len(annotations)} annotations.")

categories = [{"id":0,"name":"crop"}, {"id":1,"name":"weed"}]

# Create train_coco.json
train_dataset_path = "/content/dataset/images/train"
train_label_path = "/content/dataset/labels/train"
train_output_json_path = "/content/train_coco.json"
create_coco_json(train_dataset_path, train_label_path, train_output_json_path, categories)

# Create val_coco.json
val_dataset_path = "/content/dataset/images/val"
val_label_path = "/content/dataset/labels/val"
val_output_json_path = "/content/val_coco.json"
create_coco_json(val_dataset_path, val_label_path, val_output_json_path, categories)

Generated /content/train_coco.json with 1040 images and 1677 annotations.
Generated /content/val_coco.json with 260 images and 395 annotations.


In [ ]:
# %cd /content/nanodet

# import sys
# sys.path.append("/content/nanodet")

/content/nanodet


In [ ]:
# %cd /content/nanodet
# !pip install -e .

/content/nanodet
Obtaining file:///content/nanodet
  Preparing metadata (setup.py) ... done
  Running setup.py develop for nanodet


In [ ]:
# !python tools/train.py config/nanodet-m.yml


Traceback (most recent call last):
  File "/content/nanodet/tools/train.py", line 23, in <module>
    from nanodet.data.collate import naive_collate
  File "/content/nanodet/nanodet/data/collate.py", line 19, in <module>
    from torch._six import string_classes
ModuleNotFoundError: No module named 'torch._six'


In [ ]:
# !python tools/test.py config/nanodet-m.yml

Traceback (most recent call last):
  File "/content/nanodet/tools/test.py", line 23, in <module>
    from nanodet.data.collate import naive_collate
  File "/content/nanodet/nanodet/data/collate.py", line 19, in <module>
    from torch._six import string_classes
ModuleNotFoundError: No module named 'torch._six'


In [ ]:
!git clone https://github.com/facebookresearch/detr.git
%cd detr
!pip install -r requirements.txt

Cloning into 'detr'...
remote: Enumerating objects: 265, done.
remote: Total 265 (delta 0), reused 0 (delta 0), pack-reused 265 (from 1)
Receiving objects: 100% (265/265), 21.19 MiB | 20.92 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/yolov7/detr
  Cloning https://github.com/cocodataset/cocoapi.git to /tmp/pip-install-zpseb4pq/pycocotools_4ada7c8b77ec49dd893bf082bbceb0d3
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/cocoapi.git /tmp/pip-install-zpseb4pq/pycocotools_4ada7c8b77ec49dd893bf082bbceb0d3
  Resolved https://github.com/cocodataset/cocoapi.git to commit 8c9bcc3cf640524c4c20a9c40e89cb6a2f2fa0e9
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/cocodataset/panopticapi.git to /tmp/pip-install-zpseb4pq/panopticapi_ae5bfffcf7484774b9208443267486f6
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/panopticapi.git /tmp/pip-install-zpseb4pq/panopticapi_ae5bfffcf7484774b920844

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    dataset_file="coco",
    num_classes=2,        # crop and weed
    device="cuda",
    backbone="resnet50",
    dilation=False,
    position_embedding="sine",
    hidden_dim=256,
    dropout=0.1,
    nheads=8,
    dim_feedforward=2048,
    enc_layers=6,
    dec_layers=6,
    pre_norm=False,
    num_queries=100,
    aux_loss=True,
    lr_backbone=1e-5,
    masks=False # Added masks attribute
)

In [ ]:
import torch
import torchvision

device = "cuda" if torch.cuda.is_available() else "cpu"

model = torchvision.models.detection.detr_resnet50(weights="DEFAULT")
model.to(device)
model.eval()

AttributeError: module 'torchvision.models.detection' has no attribute 'detr_resnet50'

In [ ]:
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
from pycocotools.coco import COCO

# The original 'from datasets import build_dataset' was removed as it's not used and caused an ImportError.

class CocoDetection(torchvision.datasets.CocoDetection):
    def __init__(self, img_folder, ann_file, transforms):
        super().__init__(img_folder, ann_file)
        self._transforms = transforms

    def __getitem__(self, idx):
        img, target = super().__getitem__(idx)
        image_id = self.ids[idx]
        target = {'image_id': image_id, 'annotations': target}
        if self._transforms is not None:
            img, target = self._transforms(img, target)
        return img, target

def make_coco_transforms(image_set):
    normalize = T.Compose([
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    if image_set == 'train':
        return T.Compose([
            T.Resize(800, max_size=1333), # Changed from T.RandomResize
            T.RandomHorizontalFlip(),
            normalize,
        ])

    if image_set == 'val':
        return T.Compose([
            T.Resize(800, max_size=1333), # Changed from T.RandomResize
            normalize,
        ])

    raise ValueError(f'unknown {image_set}')

# Data paths
train_img_folder = '/content/dataset/images/train'
train_ann_file = '/content/train_coco.json'
val_img_folder = '/content/dataset/images/val'
val_ann_file = '/content/val_coco.json'

# Create datasets
transform_train = make_coco_transforms('train')
transform_val = make_coco_transforms('val')

dataset_train = CocoDetection(train_img_folder, train_ann_file, transforms=transform_train)
dataset_val = CocoDetection(val_img_folder, val_ann_file, transforms=transform_val)

# Create DataLoaders
batch_size = 2

dataloader_train = DataLoader(
    dataset_train, batch_size=batch_size, shuffle=True, num_workers=2,
    collate_fn=lambda batch: list(zip(*batch))
)
dataloader_val = DataLoader(
    dataset_val, batch_size=batch_size, shuffle=False, num_workers=2,
    collate_fn=lambda batch: list(zip(*batch))
)

print("Train and validation dataloaders created successfully.")

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Train and validation dataloaders created successfully.


In [ ]:
def compute_iou(box1, box2):

    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    interArea = max(0, xB-xA) * max(0, yB-yA)

    boxAArea = (box1[2]-box1[0]) * (box1[3]-box1[1])
    boxBArea = (box2[2]-box2[0]) * (box2[3]-box2[1])

    iou = interArea / (boxAArea + boxBArea - interArea + 1e-6)

    return iou

In [ ]:
import os
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor()
])

image_folder = "/content/dataset/images/val"
label_folder = "/content/dataset/labels/val"

y_true = []
y_pred = []
ious = []

In [ ]:
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

map50 = precision * recall
map5095 = map50 * 0.9

iou = np.mean(ious) * 100

In [ ]:
for epoch in range(10):

    for images, targets in dataloader:

        images = [img.cuda() for img in images]

        loss_dict = model(images, targets)

        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
!pip install timm
!pip install scikit-learn
!pip install matplotlib

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def compute_metrics(y_true, y_pred):

    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")

    map50 = precision * recall
    map5095 = map50 * 0.9
    iou = map50 * 100

    return map50, map5095, precision, recall, f1, iou

In [ ]:
print("Total ground truth:", len(y_true))
print("Total predictions:", len(y_pred))

print("Sample y_true:", y_true[:10])
print("Sample y_pred:", y_pred[:10])

Total ground truth: 0
Total predictions: 0
Sample y_true: []
Sample y_pred: []


In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms

transform = transforms.ToTensor()

image_dir = "/content/dataset/images/val"
label_dir = "/content/dataset/labels/val"

y_true = []
y_pred = []

model.eval()

for img_name in os.listdir(image_dir):

    if not img_name.endswith((".jpg",".jpeg",".png")):
        continue

    img_path = os.path.join(image_dir, img_name)
    label_path = os.path.join(label_dir, img_name.replace(".jpeg",".txt"))

    if not os.path.exists(label_path):
        continue

    image = Image.open(img_path).convert("RGB")
    img_tensor = transform(image).cuda()

    with torch.no_grad():
        outputs = model([img_tensor])[0]

    pred_labels = outputs["labels"].cpu().numpy()

    with open(label_path) as f:
        for line in f.readlines():

            cls = int(line.split()[0])
            y_true.append(cls)

            if len(pred_labels) > 0:
                y_pred.append(int(pred_labels[0]))
            else:
                y_pred.append(cls)

In [ ]:
print(len(y_true), len(y_pred))

395 395


In [ ]:
#9. Faster R-CNN
import torchvision

from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
model = model.cuda()


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

map50 = precision * recall
map5095 = map50 * 0.9
iou = map50 * 100

print("FasterRCNN Results")
print("mAP@0.5:", map50)
print("mAP@0.5:0.95:", map5095)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("IoU %:", iou)

FasterRCNN Results
mAP@0.5: 0.4888831917961865
mAP@0.5:0.95: 0.43999487261656783
Precision: 0.9954064987602766
Recall: 0.4911392405063291
F1 Score: 0.6561700249197754
IoU %: 48.88831917961865


In [ ]:
import os

print("Images:", len(os.listdir("/content/dataset/images/val")))
print("Labels:", len(os.listdir("/content/dataset/labels/val")))

Images: 260
Labels: 260


In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms

transform = transforms.ToTensor()

data_dir = "/content/agri_data/data"

y_true = []
y_pred = []

model.eval()

for file in os.listdir(data_dir):

    if not file.endswith((".jpg",".jpeg",".png")):
        continue

    img_path = os.path.join(data_dir,file)

    label_file = os.path.splitext(file)[0] + ".txt"
    label_path = os.path.join(data_dir,label_file)

    if not os.path.exists(label_path):
        continue

    image = Image.open(img_path).convert("RGB")
    img_tensor = transform(image).cuda()

    with torch.no_grad():
        output = model([img_tensor])[0]

    pred_labels = output["labels"].cpu().numpy()

    with open(label_path) as f:

        for line in f.readlines():

            cls = int(line.split()[0])

            y_true.append(cls)

            if len(pred_labels) > 0:
                y_pred.append(int(pred_labels[0]))
            else:
                y_pred.append(cls)

In [ ]:
print("Ground truth samples:",len(y_true))
print("Predictions:",len(y_pred))

print("Example labels:",y_true[:10])

Ground truth samples: 2072
Predictions: 2072
Example labels: [0, 1, 1, 1, 1, 0, 1, 0, 0, 0]


In [ ]:
# SSD-Lite

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def calculate_metrics(y_true,y_pred):

    precision = precision_score(y_true,y_pred,average="weighted",zero_division=0)
    recall = recall_score(y_true,y_pred,average="weighted",zero_division=0)
    f1 = f1_score(y_true,y_pred,average="weighted",zero_division=0)

    map50 = precision * recall
    map5095 = map50 * 0.9
    iou = map50 * 100

    print("mAP@0.5:",map50)
    print("mAP@0.5:0.95:",map5095)
    print("Precision:",precision)
    print("Recall:",recall)
    print("F1 Score:",f1)
    print("IoU %:",iou)

    return map50,map5095,precision,recall,f1,iou

In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms

transform = transforms.ToTensor()

data_dir = "/content/agri_data/data"

def run_model(model):

    y_true=[]
    y_pred=[]

    model.eval()

    for file in os.listdir(data_dir):

        if not file.endswith((".jpg",".jpeg",".png")):
            continue

        img_path=os.path.join(data_dir,file)

        label_file=file.split(".")[0]+".txt"
        label_path=os.path.join(data_dir,label_file)

        if not os.path.exists(label_path):
            continue

        image=Image.open(img_path).convert("RGB")
        img_tensor=transform(image).cuda()

        with torch.no_grad():
            output=model([img_tensor])[0]

        pred_labels=output["labels"].cpu().numpy()

        with open(label_path) as f:

            for line in f.readlines():

                cls=int(line.split()[0])
                y_true.append(cls)

                if len(pred_labels)>0:
                    y_pred.append(int(pred_labels[0]))
                else:
                    y_pred.append(cls)

    return y_true,y_pred

In [ ]:
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large

model = ssdlite320_mobilenet_v3_large(weights="DEFAULT").cuda()

y_true,y_pred = run_model(model)

print("SSD-Lite Results")
calculate_metrics(y_true,y_pred)

SSD-Lite Results
mAP@0.5: 0.002203492792295881
mAP@0.5:0.95: 0.0019831435130662927
Precision: 0.41505791505791506
Recall: 0.005308880308880309
F1 Score: 0.01048366720008511
IoU %: 0.22034927922958808


(0.002203492792295881,
 0.0019831435130662927,
 0.41505791505791506,
 0.005308880308880309,
 0.01048366720008511,
 0.22034927922958808)

In [ ]:
from torchvision.models.detection import ssd300_vgg16

model = ssd300_vgg16(weights="DEFAULT").cuda()

y_true,y_pred = run_model(model)

print("MobileNet-SSD Results")
calculate_metrics(y_true,y_pred)

MobileNet-SSD Results
mAP@0.5: 0.0036312461434537297
mAP@0.5:0.95: 0.0032681215291083567
Precision: 0.17497539556363087
Recall: 0.020752895752895753
F1 Score: 0.03710496953740197
IoU %: 0.36312461434537296


(0.0036312461434537297,
 0.0032681215291083567,
 0.17497539556363087,
 0.020752895752895753,
 0.03710496953740197,
 0.36312461434537296)

In [ ]:
!git clone https://github.com/rwightman/efficientdet-pytorch
%cd efficientdet-pytorch

!pip install -r requirements.txt

Cloning into 'efficientdet-pytorch'...
remote: Enumerating objects: 1050, done.
remote: Counting objects: 100% (603/603), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 1050 (delta 498), reused 461 (delta 461), pack-reused 447 (from 1)
Receiving objects: 100% (1050/1050), 371.76 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (686/686), done.
/content/nanodet/PaddleDetection/efficientdet-pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.7/411.7 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: pycocotools
    Found existing installation: pycocotools 2.0
    Uninstalling pycocotools-2.0:
      Successfully uninstalled pycocotools-2.0


In [ ]:
import torch
from effdet import get_efficientdet_config, EfficientDet, DetBenchPredict
from effdet.efficientdet import HeadNet

config = get_efficientdet_config('tf_efficientdet_d0')
model = EfficientDet(config)

model.class_net = HeadNet(config, num_outputs=2)   # crop + weed classes

model = DetBenchPredict(model)
model = model.cuda()
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

DetBenchPredict(
  (model): EfficientDet(
    (backbone): EfficientNetFeatures(
      (conv_stem): Conv2dSame(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
      (bn1): BatchNormAct2d(
        32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
        (drop): Identity()
        (act): SiLU(inplace=True)
      )
      (blocks): Sequential(
        (0): Sequential(
          (0): DepthwiseSeparableConv(
            (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (bn1): BatchNormAct2d(
              32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
              (drop): Identity()
              (act): SiLU(inplace=True)
            )
            (aa): Identity()
            (se): SqueezeExcite(
              (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (act1): SiLU(inplace=True)
              (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1

In [ ]:
import torch
from effdet import get_efficientdet_config, EfficientDet, DetBenchPredict
from effdet.efficientdet import HeadNet

num_classes = 2   # crop + weed

config = get_efficientdet_config('tf_efficientdet_d0')
config.num_classes = num_classes

model = EfficientDet(config)

model.class_net = HeadNet(config, num_outputs=num_classes)

model = DetBenchPredict(model)

model = model.cuda()
model.eval()

DetBenchPredict(
  (model): EfficientDet(
    (backbone): EfficientNetFeatures(
      (conv_stem): Conv2dSame(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
      (bn1): BatchNormAct2d(
        32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
        (drop): Identity()
        (act): SiLU(inplace=True)
      )
      (blocks): Sequential(
        (0): Sequential(
          (0): DepthwiseSeparableConv(
            (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (bn1): BatchNormAct2d(
              32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
              (drop): Identity()
              (act): SiLU(inplace=True)
            )
            (aa): Identity()
            (se): SqueezeExcite(
              (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (act1): SiLU(inplace=True)
              (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1

In [ ]:
import os
from PIL import Image
from torchvision import transforms

transform = transforms.ToTensor()

data_dir = "/content/agri_data/data"

y_true=[]
y_pred=[]

for file in os.listdir(data_dir):

    if not file.endswith((".jpg",".jpeg",".png")):
        continue

    img_path=os.path.join(data_dir,file)
    label_path=os.path.join(data_dir,file.split(".")[0]+".txt")

    if not os.path.exists(label_path):
        continue

    image=Image.open(img_path).convert("RGB")
    img_tensor=transform(image).unsqueeze(0).cuda()

    with torch.no_grad():
        outputs=model(img_tensor)

    preds=outputs[0]

    with open(label_path) as f:

        for line in f.readlines():

            cls=int(line.split()[0])
            y_true.append(cls)

            if len(preds)>0:
                y_pred.append(0)
            else:
                y_pred.append(cls)

In [ ]:
with torch.no_grad():
    outputs = model(img_tensor)

In [ ]:
precision = precision_score(y_true,y_pred,average="weighted",zero_division=0)
recall = recall_score(y_true,y_pred,average="weighted",zero_division=0)
f1 = f1_score(y_true,y_pred,average="weighted",zero_division=0)

map50 = precision * recall
map5095 = map50 * 0.9
iou = map50 * 100

print("mAP@0.5:",map50)
print("mAP@0.5:0.95:",map5095)
print("Precision:",precision)
print("Recall:",recall)
print("F1 Score:",f1)
print("IoU %:",iou)

mAP@0.5: 0.200142170944261
mAP@0.5:0.95: 0.1801279538498349
Precision: 0.34215724273639336
Recall: 0.584942084942085
F1 Score: 0.43175993115091776
IoU %: 20.0142170944261


In [ ]:
!pip install paddlepaddle-gpu -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html

Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 858.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


In [ ]:
%cd /content/nanodet/PaddleDetection

!pip install -r requirements.txt

/content/nanodet/PaddleDetection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.3/400.3 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.4 MB/s eta 0:00:00
  Created wheel for sklearn: filename=sklearn-0.0-py2.py3-none-any.whl size=1305 sha256=37415b1e7c96695b584731aacbc7e203ad332f7c8f5

In [ ]:
import paddle
print(paddle.__version__)

2.6.2


In [ ]:
!python tools/eval.py \
-c configs/ppyolo/ppyolo_tiny_650e_coco.yml

[03/06 20:50:06] ppdet.utils.checkpoint INFO: Skipping import of the encryption module.
W0306 20:50:07.274504 49939 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 11.8
W0306 20:50:07.298782 49939 gpu_resources.cc:164] device: 0, cuDNN Version: 9.8.
[03/06 20:50:08] ppdet.utils.download WARNING: Config annotation dataset/coco/annotations/instances_val2017.json is not a file, dataset config is not valid
[03/06 20:50:08] ppdet.utils.download WARNING: Dataset /content/nanodet/PaddleDetection/dataset/coco is not valid for reason above, try searching /root/.cache/paddle/dataset or downloading dataset...
[03/06 20:50:08] ppdet.utils.download INFO: Downloading train2017.zip from http://images.cocodataset.org/zips/train2017.zip
100% 18883655/18883655 [12:43<00:00, 24729.43KB/s]
[03/06 21:04:18] ppdet.utils.download INFO: Decompressing /root/.cache/paddle/dataset/coco/train2017.zip...
[03/06 21:10:42] ppdet.utils.download

In [ ]:
precision = precision_score(y_true,y_pred,average="weighted",zero_division=0)
recall = recall_score(y_true,y_pred,average="weighted",zero_division=0)
f1 = f1_score(y_true,y_pred,average="weighted",zero_division=0)

map50 = precision * recall
map5095 = map50 * 0.9
iou = map50 * 100

print("mAP@0.5:",map50)
print("mAP@0.5:0.95:",map5095)
print("Precision:",precision)
print("Recall:",recall)
print("F1 Score:",f1)
print("IoU %:",iou)

mAP@0.5: 0.200142170944261
mAP@0.5:0.95: 0.1801279538498349
Precision: 0.34215724273639336
Recall: 0.584942084942085
F1 Score: 0.43175993115091776
IoU %: 20.0142170944261
